# How the PDF chatbot works

This notebook walks through the same pipeline the website uses. You upload a PDF, the app reads its text, cuts that text into chunks, turns each chunk into numbers (an embedding), stores those numbers in a local vector database, and later sends only the closest chunks to a local language model.

Nothing here calls a paid API. There is no built-in document set. A question is answered only from PDFs that were uploaded and then selected.

Run the cells from top to bottom. Ollama must be running (`ollama serve`) with `nomic-embed-text`. The answer cells also use the local vLLM server. If vLLM is down, those cells fall back to `llama3.2:3b` on Ollama, which is slower.

The lesson indexes a tiny file named `notebook-demo.pdf`, then deletes it in the last cell. Your other uploads stay in the library.

## 1. Project setup

The website and this notebook share one Python package, `rag_chatbot`. Settings such as model names and folders live in `rag_chatbot/config.py`. The notebook adds the project root to `sys.path` so those imports work whether you started Jupyter from the project root or from `notebooks/`.

In [1]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "web_app.py").is_file():
    root = root.parent
if not (root / "web_app.py").is_file():
    raise SystemExit("Open this notebook from the project root or from notebooks/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from rag_chatbot import config
from rag_chatbot.ollama_client import chat_stream, embed_text, require_models
from rag_chatbot.uploads import (
    chunk_document_text,
    extract_pdf_pages,
    index_upload,
    prepare_document_answer,
    remove_upload,
    source_list,
)
from rag_chatbot.vllm_client import chat_stream_vllm

print("Project:", root)
print("Embeddings:", config.EMBED_MODEL, "at", config.OLLAMA_URL)
print("Answers:", config.VLLM_MODEL, "at", config.VLLM_URL)
print("Fallback chat model:", config.CHAT_MODEL)
print("Index folder:", config.UPLOAD_CHROMA_DIR)
print("Chunks kept per question:", config.TOP_K)

Project: /home/khushi/Desktop/repo/chatbot-agents
Embeddings: nomic-embed-text at http://127.0.0.1:11434
Answers: Qwen/Qwen2.5-7B-Instruct at http://127.0.0.1:8000/v1
Fallback chat model: llama3.2:3b
Index folder: /home/khushi/Desktop/repo/chatbot-agents/.chroma_uploads
Chunks kept per question: 4


## 2. Retrieval-augmented generation

A language model can write fluent text, but it does not automatically know what is inside a PDF you just uploaded. Retrieval-augmented generation (RAG) fixes that by looking up relevant passages first and pasting them into the prompt.

```
PDF text  ->  chunks  ->  embeddings  ->  vector database
question  ->  embedding  ->  closest chunks  ->  prompt  ->  local model  ->  answer
```

The lookup is the retrieval step. The written reply is the generation step. If the retrieved text does not contain the fact, the prompt tells the model to say so instead of guessing.

## 3. PDF processing

A PDF is a page layout, not a plain text file. `pypdf` reads the text layer: the selectable characters stored in the file. Each page keeps its page number, because later answers cite `filename` and `page`.

A scanned page is only a picture. This project does not run OCR, so a PDF with no text layer is rejected. An encrypted PDF that cannot be opened with an empty password is rejected too.

The next cell builds a one-page sample PDF in memory. The app does the same extraction on files you upload through the website.

In [2]:
def text_pdf(text: str) -> bytes:
    """Minimal one-page PDF. Real uploads are read the same way, with pypdf."""
    safe = text.replace("\\", "\\\\").replace("(", "\\(").replace(")", "\\)")
    stream = f"BT /F1 18 Tf 72 720 Td ({safe}) Tj ET".encode("latin-1")
    objects = [
        b"<< /Type /Catalog /Pages 2 0 R >>",
        b"<< /Type /Pages /Kids [3 0 R] /Count 1 >>",
        b"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] /Contents 4 0 R "
        b"/Resources << /Font << /F1 5 0 R >> >> >>",
        b"<< /Length %d >>\nstream\n" % len(stream) + stream + b"\nendstream",
        b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>",
    ]
    out = bytearray(b"%PDF-1.4\n")
    offsets = [0]
    for i, obj in enumerate(objects, start=1):
        offsets.append(len(out))
        out.extend(f"{i} 0 obj\n".encode())
        out.extend(obj)
        out.extend(b"\nendobj\n")
    xref = len(out)
    out.extend(f"xref\n0 {len(objects)+1}\n".encode())
    out.extend(b"0000000000 65535 f \n")
    for offset in offsets[1:]:
        out.extend(f"{offset:010d} 00000 n \n".encode())
    out.extend(
        f"trailer << /Size {len(objects)+1} /Root 1 0 R >>\nstartxref\n{xref}\n%%EOF".encode()
    )
    return bytes(out)


DEMO_NAME = "notebook-demo.pdf"
DEMO_TEXT = (
    "Amina Sharma maintains the notebook demo garden. "
    "The garden gate code is 2468."
)
pdf_bytes = text_pdf(DEMO_TEXT)
sample_path = root / "data" / "uploads" / "_notebook_extract_probe.pdf"
sample_path.parent.mkdir(parents=True, exist_ok=True)
sample_path.write_bytes(pdf_bytes)
try:
    pages = extract_pdf_pages(sample_path)
finally:
    sample_path.unlink(missing_ok=True)

pages

[{'page': 1,
  'text': 'Amina Sharma maintains the notebook demo garden. The garden gate code is 2468.'}]

## 4. Text chunking

A whole PDF can be too long to embed or to paste into one prompt. The app splits text on blank lines (paragraphs). A paragraph longer than 700 characters is cut into 700-character pieces. Each piece stays tied to the page it came from.

Short chunks are easier to match to a question. The tradeoff is that a fact split across two chunks can be missed if only one of them is retrieved. The website keeps the closest `TOP_K` chunks (4 by default) and trims each excerpt to 700 characters before it goes into the prompt.

In [3]:
page_text = pages[0]["text"]
print("Page", pages[0]["page"], "chunks:")
for i, piece in enumerate(chunk_document_text(page_text)):
    print(i, piece)

long_paragraph = "beta " * 400
long_chunks = chunk_document_text("Alpha policy.\n\n" + long_paragraph)
print("A long paragraph becomes", len(long_chunks), "chunks.")
print("First chunk starts:", long_chunks[0][:40])
print("Second chunk length:", len(long_chunks[1]))

Page 1 chunks:
0 Amina Sharma maintains the notebook demo garden. The garden gate code is 2468.
A long paragraph becomes 4 chunks.
First chunk starts: Alpha policy.
Second chunk length: 699


## 5. Embeddings

An embedding model turns a string into a long list of numbers. Strings with similar meaning land closer together than unrelated strings. This project calls Ollama's `nomic-embed-text` model at `http://127.0.0.1:11434`. The same function embeds both PDF chunks (when you upload) and the question (when you ask).

The number stored with each chunk is the embedding of the chunk text itself. Search later compares the question's embedding with those stored vectors. A smaller Chroma distance means a closer match. The app reports similarity as `1 - distance`.

In [4]:
require_models(need_chat=False)

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = sum(x * x for x in a) ** 0.5
    nb = sum(y * y for y in b) ** 0.5
    return dot / (na * nb)

garden = embed_text("The garden gate code is 2468.")
related = embed_text("What is the code for the garden gate?")
unrelated = embed_text("How do I bake sourdough bread?")

print("Embedding length:", len(garden))
print("First values:", [round(v, 4) for v in garden[:8]])
print("Related question cosine:", round(cosine(garden, related), 3))
print("Unrelated question cosine:", round(cosine(garden, unrelated), 3))

Embedding length: 768
First values: [0.1281, 0.0995, -3.715, 0.233, 1.1149, 0.1512, 0.3699, 0.7225]
Related question cosine: 0.869
Unrelated question cosine: 0.362


## 6. Vector database

Chroma stores the chunk text, its embedding, and metadata on disk under `.chroma_uploads/`. The collection is named `user_uploads` and uses cosine distance.

Each chunk is saved with:

- `source`: the PDF filename
- `page`: the page number
- `chunk_id`: the position inside that file
- `sha256`: a fingerprint of the uploaded bytes

`data/uploads/manifest.json` records the filename, fingerprint, page count, and chunk count. Uploading the same bytes again does not create a second copy. Removing a file deletes the PDF and its chunks. You do not rebuild a separate Harbor index, and you do not restart the server.

The next cell calls `index_upload`, which is the function behind `POST /api/upload`.

In [5]:
remove_upload(DEMO_NAME)
indexed = index_upload(DEMO_NAME, pdf_bytes)
indexed

{'name': 'notebook-demo.pdf',
 'chunks': 1,
 'pages': 1,
 'duplicate': False,
 'status': 'ready'}

## 7. Retrieval

Asking a question does not scan every PDF on disk. `prepare_document_answer` keeps only the filenames you selected, embeds the question, and asks Chroma for the closest chunks whose `source` is one of those files. Chunks from other uploads are left out.

If the selection is empty, there is no model call. The reply is: "Select at least one indexed PDF before asking a question."

The retrieved excerpts are wrapped as `CONTEXT`, with the filename and page on each block. That package, plus the question, is the user message. The system message tells the model to use only that context, cite the file and page, and admit when the fact is missing.

In [6]:
empty = prepare_document_answer("What is the garden gate code?", [])
print("No selection:", empty["answer"])

prepared = prepare_document_answer("What is the garden gate code?", [DEMO_NAME])
print("Sources:", source_list(prepared["hits"]))
for hit in prepared["hits"]:
    print(
        f"{hit['source']} page {hit['page']}  "
        f"distance={hit['distance']:.3f}  similarity={hit['similarity']:.3f}"
    )
print()
print(prepared["messages"][1]["content"])

No selection: Select at least one indexed PDF before asking a question.


Sources: [{'name': 'notebook-demo.pdf', 'page': 1}]
notebook-demo.pdf page 1  distance=0.185  similarity=0.815

CONTEXT:
[1] (notebook-demo.pdf p.1)
Amina Sharma maintains the notebook demo garden. The garden gate code is 2468.

QUESTION:
What is the garden gate code?


## 8. Local language model

Generation stays on this machine. The first choice is vLLM, an OpenAI-compatible server at `http://127.0.0.1:8000/v1`, model `Qwen/Qwen2.5-7B-Instruct`. The client sends `temperature` and `max_tokens` from `config.py` and reads the streamed tokens.

If that request fails, the app streams the same messages through Ollama's chat API using `CHAT_MODEL` (`llama3.2:3b`). Embeddings always use Ollama, even when the answer comes from vLLM.

The website does this inside `POST /api/chat`: prepare the messages once, stream tokens to the browser, then send the source list.

In [7]:
def stream_answer(messages):
    """Same order as web_app.py: vLLM first, Ollama chat if vLLM fails."""
    try:
        yield from chat_stream_vllm(messages)
        return
    except Exception as exc:
        print(f"\n[vLLM unavailable: {exc}. Falling back to {config.CHAT_MODEL}.]\n")
    yield from chat_stream(messages)

answer = "".join(stream_answer(prepared["messages"]))
print(answer)
print("Cited sources:", source_list(prepared["hits"]))

The garden gate code is 2468. [1] (notebook-demo.pdf p.1)
Cited sources: [{'name': 'notebook-demo.pdf', 'page': 1}]


## 9. A question the PDF does not answer

Retrieval still returns the closest chunk, even when the question is off topic. The prompt then tells the model that the context is the only allowed source. The reply should say the fact is not in the selected document, rather than inventing it from general knowledge.

In [8]:
missing = prepare_document_answer("What is the capital of France?", [DEMO_NAME])
print(missing["messages"][1]["content"])
print()
print("".join(stream_answer(missing["messages"])))

CONTEXT:
[1] (notebook-demo.pdf p.1)
Amina Sharma maintains the notebook demo garden. The garden gate code is 2468.

QUESTION:
What is the capital of France?



The capital of France is not in the selected documents.


## 10. What the website adds

The React page is the way a person drives this pipeline:

1. **Upload PDF** sends the file to `POST /api/upload`, which calls `index_upload`.
2. The library lists ready files from `GET /api/files`. Checking a box selects it. The question box stays off until at least one ready PDF is selected.
3. Send calls `POST /api/chat` with `{ "question", "documents" }`. `documents` is only the checked filenames.
4. Tokens stream into the assistant bubble. The source chips are the filename and page from `source_list`.
5. **Remove** calls `DELETE /api/files/<name>`.

Flask serves the built page from `frontend/dist` at http://127.0.0.1:5000. The browser does not embed text or talk to Ollama itself.

## 11. Clean up the lesson file

Run this cell when you are finished. It deletes `notebook-demo.pdf` and its vectors. Other uploaded PDFs are left alone.

In [9]:
remove_upload(DEMO_NAME)
print("Removed", DEMO_NAME)

Removed notebook-demo.pdf
